In [ ]:
import numpy as np
import psfsim
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy.interpolate as sp_itp

from psfsim.polychrom import PolychromaticPSF

In [ ]:
j129 = np.linspace(1.131, 1.454, 10)  # microns

# PSF generation and downsampling parameters
ps_size = 96
ovsamp = pixel_size = 8
mag = 1e8  # some unit

# Poisson resampling and spike fitting parameters
step = 0.1  # degrees
borders = (0.2, 1.0)
bound = ps_size // 2
center = np.array((ps_size // 2, ps_size // 2))
flux_factor = pixel_size**2 * mag

# Interpolation parameters
dense_ps_size = 600
dense_bound = dense_ps_size // 2
dense_center = np.array((dense_ps_size // 2, dense_ps_size // 2))

In [ ]:
def line_coords(angle, bound, center=np.array((0, 0))):
    # since (0, 0) in data coords is top left instead of bottom left we must use clockwise rotations to get
    # a visually ccl rotation. However we will take a transpose later anyways so the rotation matrix def is ccl, as expected. How quaint
    angle = np.deg2rad(angle)
    # print( angle )
    line = np.zeros((2, bound))
    line[0] = np.arange(bound)

    rotation = np.array(
        (
            (np.cos(angle), -np.sin(angle)),
            (np.sin(angle), np.cos(angle)),
        )
    )

    if len(rotation.shape) > 2:
        line = line.reshape((1, 2, bound))

    # print( line.shape )
    # print( rotation.T.shape )

    return np.int32(rotation.T @ line) + np.int32(np.reshape(center, (2, 1)))


def find_spikes(
    image, step, bound=1024, center=np.array((0, 0)), borders=(0.0, 1.0), threshold=0.5, verbose=False
):
    angles = np.linspace(0, 360, num=np.int32(360.0 / step), endpoint=False)

    lines = line_coords(angles, bound, center)
    image_analyze = (
        image.T
    )  # imshow will display the image as row,col with 0,0 in top left. For our analysis we want (col,row).

    line_vals = image_analyze[lines[:, 0], lines[:, 1]]
    sums = np.sum(line_vals, axis=-1)

    # 0 to 1 for these. Should I add a way to make it force 0 <= borders[0] < borders[1] <= 1?
    range_low = np.int_(borders[0] * bound)
    range_high = np.int_(borders[1] * bound)

    sums = np.sum(line_vals[:, range_low:range_high], axis=-1)
    """
    This formula is slightly different from the midpoint formula from before. The formula defaults to 50%, using 50% recovers the original behavior. Other
    ways to do this include rejecting the lowest 10% of values or so. That one might be worth pursuing in the future. The goal is to keep this method as simple as possible--
    I do NOT want to start staring at noise to find spikes.

    There is another method that may be worth pursuing that involves a boxcar average that goes around the circle and might be able to correct for groups of spikes that are much
    higher or lower than the naive cutoff assumes.
    """
    diff = np.max(sums) - np.min(sums)
    cutoff = np.min(sums) + (diff * threshold)
    spike_indices = np.where(sums > cutoff)[0]
    spike_angles = angles[spike_indices]
    # print( np.diff( spike_angles ) )

    # try 3 degrees to tell differing spikes apart for now
    spike_angle_discriminator = 3.0
    borders = np.where(np.diff(spike_angles) > spike_angle_discriminator)[0]
    spike_groups = np.split(spike_angles, borders + 1)
    spike_group_indices = np.split(spike_indices, borders + 1)

    print(f"Cutoff of {cutoff:.3f}")
    # unfortunately this for loop is necessary unless theres a convenient package for jagged arrays
    spike_list = np.zeros(len(spike_groups))
    for i in np.arange(len(spike_groups)):
        idx_min_r = spike_group_indices[i][0]
        idx_max_l = spike_group_indices[i][-1]

        idx_min_l = idx_min_r - 1
        idx_max_r = idx_max_l + 1

        slope_left = (sums[idx_min_r] - sums[idx_min_l]) / step
        slope_right = (sums[idx_max_r] - sums[idx_max_l]) / step

        th_min = ((cutoff - sums[idx_min_l]) / slope_left) + angles[idx_min_l]
        th_max = ((cutoff - sums[idx_max_l]) / slope_right) + angles[idx_max_l]

        # print( f'{sums[idx_min_l]:.3f}, {sums[idx_min_r]:.3f} and {sums[idx_max_l]:.3f}, {sums[idx_max_r]:.3f}' )
        print(
            f"{angles[idx_min_l]:.2f}, {angles[idx_min_r]:.2f} and {angles[idx_max_l]:.2f}, {angles[idx_max_r]:.2f}. interpolated as {th_min:.2f}, {th_max:.2f}"
        )

        # spike_list[i] = np.median( spike_groups[i] )
        spike_list[i] = (th_min + th_max) / 2.0

    # print( spike_list )
    if verbose:
        return sums, spike_angles, cutoff, spike_list
    else:
        return spike_list


def draw_ray(ax, angle, bound, center=np.array((0, 0)), borders=(0.0, 1.0), **kwargs):
    line = line_coords(angle, bound, center)
    range_low = np.int_(borders[0] * bound)
    range_high = np.int_(borders[1] * bound)
    ax.plot(line[0, range_low:range_high], line[1, range_low:range_high], **kwargs)
    return


def downsample_2d_image(image, pixel_size=8):  # ONLY WORKS ON 2D IMAGES
    subpixel_side = np.arange(pixel_size)
    subpixel_cols, subpixel_rows = np.meshgrid(subpixel_side, subpixel_side)

    dsamp_size = image.shape[0] // pixel_size
    pixels_side = np.arange(dsamp_size) * pixel_size
    pixels_cols, pixels_rows = np.meshgrid(pixels_side, pixels_side)

    pixels_rows = pixels_rows[:, :, np.newaxis]  # promote to 3D
    pixels_cols = pixels_cols[:, :, np.newaxis]

    idx_arr_subpixel_rows = np.full((dsamp_size, dsamp_size, pixel_size**2), subpixel_rows.flatten())
    idx_arr_subpixel_cols = np.full((dsamp_size, dsamp_size, pixel_size**2), subpixel_cols.flatten())

    idx_arr_rows = idx_arr_subpixel_rows + pixels_rows
    idx_arr_cols = idx_arr_subpixel_cols + pixels_cols

    pixelated_image = image[idx_arr_rows, idx_arr_cols]
    downsampled_image = np.mean(pixelated_image, axis=-1)
    return downsampled_image


def poisson_resample_image(img, flux_factor):
    # resamples an image where each pixel gets its own poisson distribution
    # this will change the scales around so that the resulting image has large positive values!!!! be aware
    rng = np.random.default_rng()
    poisson_means = img * flux_factor

    resampled_img = rng.poisson(poisson_means, size=poisson_means.shape)
    return resampled_img


def interpolate_image(img, dense_size):  # 2D ONLY
    x_pts = np.arange(img.shape[0])
    y_pts = np.arange(img.shape[1])

    new_grid = sp_itp.RegularGridInterpolator((x_pts, y_pts), img, method="linear")
    dense_x_1d = np.linspace(0, img.shape[0] - 1, dense_size)
    dense_y_1d = np.linspace(0, img.shape[1] - 1, dense_size)
    dense_x, dense_y = np.meshgrid(dense_x_1d, dense_y_1d, indexing="ij")

    return new_grid((dense_x, dense_y))

In [ ]:
def compute_response_matrix(aberrations, ideal_spike_angles, psf_object, step, bound, center, **kwargs):
    resp_matrix = np.zeros((12, 5))
    for i in np.arange(aberrations.size):
        extra_aberrations = [None, None, None, None, None]
        extra_aberrations[i] = aberrations[i]

        psf_object.compute_poly_psf(
            postage_stamp_size=32,
            optical_psf_only=True,
            use_postage_stamp_size=96,
            ovsamp=8,
            extra_aberrations=extra_aberrations,
        )

        psf_ovsamp = np.log10(np.abs(psf_object.chromatic_psf))
        psf_dsamp = downsample_2d_image(psf_ovsamp, pixel_size=8)
        psf = interpolate_image(psf_dsamp, dense_ps_size)

        spike_list = find_spikes(psf, step, bound, center, **kwargs)
        resp_matrix[:, i] = (spike_list - ideal_spike_angles) / aberrations[i]

    return resp_matrix

In [ ]:
j129_obj = PolychromaticPSF(9, 0, 0, j129)

j129_obj.compute_poly_psf(
    optical_psf_only=True, use_postage_stamp_size=ps_size, ovsamp=ovsamp, use_filter="J129"
)
psf_ideal = interpolate_image(
    np.arcsinh(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    ),
    dense_ps_size,
)

ideal_spikes = find_spikes(psf_ideal, step, dense_bound, dense_center, borders=borders)
print(ideal_spikes)

In [ ]:
aberrations = np.full(5, 0.04)
resp_matrix_04 = compute_response_matrix(
    aberrations, ideal_spikes, j129_obj, step, dense_bound, dense_center, borders=borders
)

aberrations = np.full(5, 0.07)
resp_matrix_07 = compute_response_matrix(
    aberrations, ideal_spikes, j129_obj, step, dense_bound, dense_center, borders=borders
)

aberrations = np.full(5, 0.1)
resp_matrix_10 = compute_response_matrix(
    aberrations, ideal_spikes, j129_obj, step, dense_bound, dense_center, borders=borders
)

aberrations = np.full(5, 0.13)
resp_matrix_13 = compute_response_matrix(
    aberrations, ideal_spikes, j129_obj, step, dense_bound, dense_center, borders=borders
)

In [ ]:
rng = np.random.default_rng()

# test_aberrations = [ 0.085, 0, 0, 0.1, 0 ]
test_aberrations = (rng.random(5) * 0.4) - 0.2
# test_aberrations[ 1: ] = 0
print(f"Test Aberration Values: {test_aberrations}")


j129_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=96,
    ovsamp=ovsamp,
    use_filter="J129",
    extra_aberrations=test_aberrations,
)
test_psf = interpolate_image(
    np.arcsinh(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    ),
    dense_ps_size,
)

In [ ]:
big_resp_matrix = np.zeros((12, 5, 4))
big_resp_matrix[:, :, 0] = resp_matrix_04
big_resp_matrix[:, :, 1] = resp_matrix_07
big_resp_matrix[:, :, 2] = resp_matrix_10
big_resp_matrix[:, :, 3] = resp_matrix_13
resp_matrix = np.mean(big_resp_matrix, axis=-1)


predict_angle_diffs = resp_matrix @ np.array(test_aberrations)
spike_list_test = find_spikes(test_psf, step, dense_bound, dense_center, borders=borders)

dth_vec = spike_list_test - ideal_spikes
predict_aberrations = np.linalg.inv(resp_matrix.T @ resp_matrix) @ resp_matrix.T @ dth_vec
N = np.arange(spike_list_test.size)

print(f"Generated aberrations: {test_aberrations}")
print(f"Predicted from response matrix: {predict_aberrations}")

fig, ax = plt.subplots(figsize=(12, 6), ncols=2)

ax[0].scatter(N, predict_angle_diffs, label="response matrix prediction")
ax[0].scatter(N, dth_vec, label="measured spikes")
ax[0].axhline(0, color="C3", ls="--")
ax[0].legend()

im = ax[1].imshow(test_psf)
fig.colorbar(im, ax=ax[1])

# with np.printoptions( precision=4, floatmode='fixed' ):
#     ax[1].set_title( f'{test_aberrations}' )

for s in spike_list_test:
    draw_ray(ax[1], s, dense_bound, dense_center, borders=borders, color="C3")

Some stats about how good the response matrix estimation of the zernikes is

In [ ]:
plt.scatter(N, predict_angle_diffs - dth_vec)
plt.axhline(0, color="C3", ls="--")

In [ ]:
n = 15
N = np.full((15, 12), np.arange(12)[np.newaxis, :])
print(N)

In [ ]:
n = 15
aberration_arr = np.linspace(0.01, 0.15, n)
dth_mat = np.zeros((15, 12))
for i in np.arange(n):
    j129_obj.compute_poly_psf(
        optical_psf_only=True,
        use_postage_stamp_size=96,
        ovsamp=ovsamp,
        use_filter="J129",
        extra_aberrations=(aberration_arr[i],),
    )
    psf = np.log10(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    )
    spikes = find_spikes(psf, step, bound, center, borders=borders)
    dth_mat[i] = spikes - ideal_spikes

In [ ]:
print(dth_mat)

fig, ax = plt.subplots()

spike_count = 12

a = np.arange(n)
ax.scatter(a, dth_mat[:, 1])
ax.scatter(a, dth_mat[:, 3])
ax.scatter(a, dth_mat[:, 4])
ax.scatter(a, dth_mat[:, 5])
ax.scatter(a, dth_mat[:, 7])
ax.scatter(a, dth_mat[:, 9])
ax.scatter(a, dth_mat[:, 11])

# for i in np.arange( n )[:4]:
#     ax.scatter( np.arange( spike_count ), dth_mat[i] )

In [ ]:
n = 15
aberration_arr = np.linspace(0.01, 0.15, n)
dth_mat_vcenter = np.zeros((15, 12))
for i in np.arange(n):
    j129_obj.compute_poly_psf(
        optical_psf_only=True,
        use_postage_stamp_size=96,
        ovsamp=ovsamp,
        use_filter="J129",
        extra_aberrations=(
            None,
            aberration_arr[i],
        ),
    )
    psf = np.log10(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    )
    spikes = find_spikes(psf, step, bound, center, borders=borders)
    dth_mat_vcenter[i] = spikes - ideal_spikes

In [ ]:
print(dth_mat_vcenter)

fig, ax = plt.subplots()

spike_count = 12

a = np.arange(n)
ax.scatter(a, dth_mat_vcenter[:, 1])
ax.scatter(a, dth_mat_vcenter[:, 3])
ax.scatter(a, dth_mat_vcenter[:, 4])
ax.scatter(a, dth_mat_vcenter[:, 5])
ax.scatter(a, dth_mat_vcenter[:, 7])
ax.scatter(a, dth_mat_vcenter[:, 9])
ax.scatter(a, dth_mat_vcenter[:, 11])

In [ ]:
n = 15
aberration_arr = np.linspace(0.01, 0.15, n)
dth_mat_focus = np.zeros((15, 12))
for i in np.arange(n):
    j129_obj.compute_poly_psf(
        optical_psf_only=True,
        use_postage_stamp_size=96,
        ovsamp=ovsamp,
        use_filter="J129",
        extra_aberrations=(
            None,
            None,
            aberration_arr[i],
        ),
    )
    psf = np.log10(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    )
    spikes = find_spikes(psf, step, bound, center, borders=borders)
    dth_mat_focus[i] = spikes - ideal_spikes

In [ ]:
print(dth_mat_focus)

fig, ax = plt.subplots()

spike_count = 12

a = np.arange(n)
ax.scatter(a, dth_mat_focus[:, 1])
ax.scatter(a, dth_mat_focus[:, 3])
ax.scatter(a, dth_mat_focus[:, 4])
ax.scatter(a, dth_mat_focus[:, 5])
ax.scatter(a, dth_mat_focus[:, 7])
ax.scatter(a, dth_mat_focus[:, 9])
ax.scatter(a, dth_mat_focus[:, 11])

In [ ]:
n = 15
aberration_arr = np.linspace(0.01, 0.15, n)
dth_mat_z5 = np.zeros((15, 12))
for i in np.arange(n):
    j129_obj.compute_poly_psf(
        optical_psf_only=True,
        use_postage_stamp_size=96,
        ovsamp=ovsamp,
        use_filter="J129",
        extra_aberrations=(
            None,
            None,
            None,
            aberration_arr[i],
        ),
    )
    psf = np.log10(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    )
    spikes = find_spikes(psf, step, bound, center, borders=borders)
    dth_mat_z5[i] = spikes - ideal_spikes

In [ ]:
print(dth_mat_z5)

fig, ax = plt.subplots()

spike_count = 12

a = np.arange(n)
ax.scatter(a, dth_mat_z5[:, 1])
ax.scatter(a, dth_mat_z5[:, 3])
ax.scatter(a, dth_mat_z5[:, 4])
ax.scatter(a, dth_mat_z5[:, 5])
ax.scatter(a, dth_mat_z5[:, 7])
ax.scatter(a, dth_mat_z5[:, 9])
ax.scatter(a, dth_mat_z5[:, 11])

In [ ]:
n = 15
aberration_arr = np.linspace(0.01, 0.15, n)
dth_mat_z6 = np.zeros((15, 12))
for i in np.arange(n):
    j129_obj.compute_poly_psf(
        optical_psf_only=True,
        use_postage_stamp_size=96,
        ovsamp=ovsamp,
        use_filter="J129",
        extra_aberrations=(None, None, None, None, aberration_arr[i]),
    )
    psf = np.log10(
        poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
    )
    spikes = find_spikes(psf, step, bound, center, borders=borders)
    dth_mat_z6[i] = spikes - ideal_spikes

In [ ]:
print(dth_mat_z6)

fig, ax = plt.subplots()

spike_count = 12

a = np.arange(n)
ax.scatter(a, dth_mat_z6[:, 1])
ax.scatter(a, dth_mat_z6[:, 3])
ax.scatter(a, dth_mat_z6[:, 4])
ax.scatter(a, dth_mat_z6[:, 5])
ax.scatter(a, dth_mat_z6[:, 7])
ax.scatter(a, dth_mat_z6[:, 9])
ax.scatter(a, dth_mat_z6[:, 11])